# The Bayesian Finite Element Method in Inverse Problems: Pullout Test

This notebook is associated with section 3.1 of "The Bayesian Finite Element Method in Inverse Problems: a Critical Comparison between Probabilistic Models for Discretization Error" by Anne Poot, Iuri Rocha, Pierre Kerfriden and Frans van der Meer ([doi:10.48550/arXiv.2506.02815](https://doi.org/10.48550/arXiv.2506.02815)).

In [ ]:
# general imports
import os
import numpy as np
import urllib.request
import zipfile

# local imports
from bfem.observation import compute_bfem_observations
from fem.jive import CJiveRunner
from fem.meshing import (
    mesh_interval_with_line2,
    create_phi_from_globdat,
    calc_elem_sizes,
    calc_boundary_nodes,
)
from probability.process import (
    GaussianProcess,
    InverseCovarianceOperator,
    ProjectedPrior,
)
from rmfem.perturbation import calc_perturbed_coords
from util.io import read_csv_from

from experiments.reproduction.inverse.pullout_bar.props import get_fem_props
from experiments.reproduction.inverse.pullout_bar.plots import (
    exact_plot,
    bfem_plot,
    rmfem_plot,
    scatter_plot,
)

## Forward problem

We first consider the FEM, BFEM and RM-FEM solutions to the forward problem.

### Figure 3.1b: FEM solutions

This figure shows the FEM solution to the pullout test problem for various mesh densities, along with the exact solution.

In [ ]:
n_elems = np.array([1, 2, 4, 8, 16, 32, 64])
us = []

# get fem settings
props = get_fem_props()

# compute fem solution for each mesh size
for n_elem in n_elems:
    nodes, elems = mesh_interval_with_line2(n=n_elem)
    jive = CJiveRunner(props, elems=elems)
    globdat = jive()
    us.append(globdat["state0"])

# plot the results
exact_plot(n_elems, us)

### Figures 3.2a and 3.2b: BFEM solution

The first figure shows the BFEM prior on a reference mesh of 8 elements.
The second figure show the BFEM posterior after conditioning on an observation mesh of 4 elements.

In [ ]:
n_elem = 4
n_elem_ref = 2 * n_elem

# generate meshes
obs_nodes, obs_elems = mesh_interval_with_line2(n=n_elem)
ref_nodes, ref_elems = mesh_interval_with_line2(n=n_elem_ref)

# determine scale by maximizing marginal likelihood
jive = CJiveRunner(get_fem_props(), elems=obs_elems)
globdat = jive()
u_obs = globdat["state0"]
K_obs = globdat["matrix0"]
n_obs = len(u_obs)
alpha2_mle = u_obs @ K_obs @ u_obs / n_obs

# set up fem solver
module_props = get_fem_props()
model_props = module_props.pop("model")
ref_jive_runner = CJiveRunner(module_props, elems=ref_elems)
obs_jive_runner = CJiveRunner(module_props, elems=obs_elems)

# define prior on exact, reference and observation level
inf_cov = InverseCovarianceOperator(model_props=model_props, scale=alpha2_mle)
inf_prior = GaussianProcess(None, inf_cov)
ref_prior = ProjectedPrior(prior=inf_prior, jive_runner=ref_jive_runner)
obs_prior = ProjectedPrior(prior=inf_prior, jive_runner=obs_jive_runner)

# condition on observation shape functions
H_obs, f_obs = compute_bfem_observations(obs_prior, ref_prior)
ref_posterior = ref_prior.condition_on(H_obs, f_obs)

# plot the results
bfem_plot(ref_prior)
bfem_plot(ref_posterior)

### Figure 3.2c: RM-FEM solution

This figure shows the RM-FEM solution for a 4-element mesh.

In [ ]:
n_elem = 4
n_sample = 20  # in the paper, we use n_sample = 1000, but this takes some time

# generate meshes
ref_nodes, ref_elems = mesh_interval_with_line2(n=n_elem)
ref_coords = ref_nodes.get_coords()
elem_sizes = calc_elem_sizes(ref_elems)
boundary = calc_boundary_nodes(ref_elems)
pert_nodes, pert_elems = mesh_interval_with_line2(n=n_elem)

# set up fem solver
props = get_fem_props()
jive = CJiveRunner(props, elems=pert_elems)

rng = np.random.default_rng(0)
us = []
xs = []

for _ in range(n_sample):
    # compute perturbed coordinates
    pert_coords = calc_perturbed_coords(
        ref_coords=ref_coords,
        elems=ref_elems,
        elem_sizes=elem_sizes,
        p=1,
        boundary=boundary,
        rng=rng,
    )
    pert_nodes._data[:, :] = pert_coords
    jive.update_elems(pert_elems)

    # compute and store perturbed solution
    globdat_pert = jive()
    xs.append(globdat_pert["nodeSet"])
    us.append(globdat_pert["state0"])

us = np.array(us)
xs = np.array(xs)

# plot results
rmfem_plot(xs, us)

## Inverse Problem

We now consider the inverse problem, described in section 3.1.1.
The dataset can be regenerated, but it it easier to download it directly.

In [ ]:
cwd = os.getcwd()
tmp_path = os.path.join(cwd, "tmp")
zip_path = os.path.join(tmp_path, "pullout-bar.zip")
output_path = os.path.join(tmp_path, "output")
url = "https://data.4tu.nl/file/a610235b-7e45-4d8f-8a0b-64d8eb157b36/16f192b1-aa24-454e-8f8e-e4b325a84e2d"

if not os.path.exists(tmp_path):
    # download zip file
    os.mkdir(tmp_path)
    out = urllib.request.urlretrieve(url, zip_path)

    # extract in tmp folder
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(tmp_path)

assert os.path.isfile(zip_path)
assert os.path.isdir(output_path)

### Figure 3.3: inverse problem solutions

These figures show samples from the FEM, BFEM, RM-FEM and statFEM posteriors of the inverse problem from section 3.1.1 of the paper.

In [ ]:
# rng seed
# seed = 0  # single run
seed = "0-20"  # meta run

if isinstance(seed, int):
    # for the standard run, a burn-in period is needed during which the proposal is adapted
    N_burn = 10000
    N_filter = 50
elif isinstance(seed, str):
    # for the meta run, the proposal is never adapted, so no burn-in period is needed
    N_burn = 25
    N_filter = 50
else:
    assert False

for fem_type in ["fem", "bfem", "rmfem", "statfem"]:
    if fem_type == "fem":
        n_elem_range = [1, 2, 4, 8, 16, 32, 64]
    else:
        n_elem_range = [1, 4, 16, 64]

    # load data, discard burn-in and thin samples
    fname = os.path.join(output_path, "samples-{}_seed-{}.csv".format(fem_type, seed))
    df = read_csv_from(fname, "log_E,log_k")
    df = df[(df["sample"] >= N_burn) & (df["sample"] % N_filter == 0)]
    df = df[df["n_elem"].isin(n_elem_range)]

    # plot the posteriors
    scatter_plot(df)